In [4]:
import pandas as pd
import numpy as np
import surprise

from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [9]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\filmtrust\ratings.txt",
                 sep=' ',names=['UserId','ItemID','Rating'])
df.sample(5)

,UserId,ItemID,Rating
35028,1491,1711,1.5
3043,147,2,4.0
33722,1444,243,0.5
9575,391,211,3.5
19182,797,8,3.5


In [10]:
min_rate = df['Rating'].min()
max_rate = df['Rating'].max()

In [11]:
reader = surprise.Reader(rating_scale=(min_rate,max_rate))
data = surprise.Dataset.load_from_df(df,reader)

In [12]:
param_grid = {  'n_epochs':[50],
                'n_factors':[100,50],
                'reg_all':np.linspace(0.001,1,5),
                'reg_all':np.linspace(0.01,0.8,5)
             }
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.SVD, param_grid, measures=['rmse','mae'], joblib_verbose=3, cv=kfold, n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  36 out of  50 | elapsed:    5.3s remaining:    2.0s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    5.9s finished


In [13]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])
print(gs.best_score['mae'])
print(gs.best_params['mae'])

0.8019704601429861
{'n_epochs': 50, 'n_factors': 50, 'reg_all': np.float64(0.20750000000000002)}
0.6232672294731362
{'n_epochs': 50, 'n_factors': 50, 'reg_all': np.float64(0.20750000000000002)}


# Exercise

In [29]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\ml-100k\u.data",
                 sep='\t',names=['UserId','ItemID','Rating','TS'])
df.sample(5)

,UserId,ItemID,Rating,TS
8099,236,756,1,890117353
41061,382,122,3,875946440
29315,416,70,5,893213019
53992,109,834,3,880583308
19992,450,2,4,882474001


In [30]:
df.drop('TS',axis=1,inplace=True)

In [31]:
min_rate = df['Rating'].min()
max_rate = df['Rating'].max()
reader = surprise.Reader(rating_scale=(min_rate,max_rate))
data = surprise.Dataset.load_from_df(df,reader)

In [32]:
param_grid = {  'n_epochs':[50],
                'n_factors':[100,50],
                'reg_all':np.linspace(0.001,1,5),
                'reg_all':np.linspace(0.01,0.8,5)
             }
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.SVD, param_grid, measures=['rmse','mae'], joblib_verbose=3, cv=kfold, n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  36 out of  50 | elapsed:   12.9s remaining:    5.0s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   15.2s finished


In [33]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])
print(gs.best_score['mae'])
print(gs.best_params['mae'])

0.9432754260141657
{'n_epochs': 50, 'n_factors': 100, 'reg_all': np.float64(0.20750000000000002)}
0.7508658553777667
{'n_epochs': 50, 'n_factors': 100, 'reg_all': np.float64(0.20750000000000002)}


In [34]:
model = gs.best_estimator['rmse']
model.fit(data.build_full_trainset())

In [35]:
ItemID = df['ItemID'].unique()
len(ItemID)

1682

In [36]:
user = 50
u_iid = df[df['UserId'] == user ]['ItemID'].unique()
print('List of item rated by user :\n',u_iid)
print('No. of items rated by user {0} : {1}'.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(ItemID,u_iid)
print('Items not rated by the user or those items which the expected ratings are to be predicted :\n',iids_to_predict)

List of item rated by user :
 [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]
No. of items rated by user 50 : 24
Items not rated by the user or those items which the expected ratings are to be predicted :
 [   1    2    3 ... 1680 1681 1682]


In [37]:
testset = [[user,iid,0.] for iid in iids_to_predict]
prediction = model.test(testset)

In [38]:
exp_rating = [ (prediction[i].iid,prediction[i].est) for i in range(0,len(prediction)) ]
df_rating = pd.DataFrame(exp_rating , columns=['ItemID','Est_Rating'])
df_rating.sort_values('Est_Rating',ascending=False).head(30)

,ItemID,Est_Rating
1424,1449,4.412324
391,408,4.347131
162,169,4.317034
305,318,4.290364
61,64,4.265342
465,483,4.246400
110,114,4.246249
10,12,4.224585
582,603,4.200382
171,178,4.193017


In [39]:
movies = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\ml-100k\movies_list.csv",
                    encoding='latin-1')
movies.head()

,movie id,movie title,release date,video release date,IMDb URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [40]:
movies.columns = movies.columns.str.strip()
movies.columns 

Index(['movie id', 'movie title', 'release date', 'video release date',
       'IMDb URL', 'unknown', 'Action', 'Adventure', 'Animation', 'Children's',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')

In [41]:
rating = df_rating.merge(movies[['movie id','movie title','release date']],
                              how ='left',left_on='ItemID',right_on='movie id')
rating.sort_values('Est_Rating',ascending=False).head(10)

,ItemID,Est_Rating,movie id,movie title,release date
1424,1449,4.412324,1449,Pather Panchali (1955),22-Mar-96
391,408,4.347131,408,"Close Shave, A (1995)",28-Apr-96
162,169,4.317034,169,"Wrong Trousers, The (1993)",01-Jan-93
305,318,4.290364,318,Schindler's List (1993),01-Jan-93
61,64,4.265342,64,"Shawshank Redemption, The (1994)",01-Jan-94
465,483,4.246400,483,Casablanca (1942),01-Jan-42
110,114,4.246249,114,Wallace & Gromit: The Best of Aardman Animatio...,05-Apr-96
10,12,4.224585,12,"Usual Suspects, The (1995)",14-Aug-95
582,603,4.200382,603,Rear Window (1954),01-Jan-54
171,178,4.193017,178,12 Angry Men (1957),01-Jan-57


# Exercise

In [45]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\Amazon Movie Ratings\Amazon.csv")
rating = pd.melt(df, id_vars='user_id', 
                  var_name="item_id",value_name="rating")

rating = rating[rating['rating'].notna()]
rating.head()

,user_id,item_id,rating
0,A3R5OBKS7OM2IR,Movie1,5.0
4848,A3R5OBKS7OM2IR,Movie2,5.0
9697,AH3QC2PC1VTGP,Movie3,2.0
14546,A3LKP6WPMP9UKX,Movie4,5.0
14547,AVIY68KEPQ5ZD,Movie4,5.0


In [49]:
min_rating = rating['rating'].min()
max_rating = rating['rating'].max()

In [53]:
reader = surprise.Reader(rating_scale=(min_rating,max_rating))
data = surprise.Dataset.load_from_df(rating,reader)

In [54]:
# Using KNNBasis
param_grid = { 'k':[25,40,50,65,75,85],
             'user_based':[False,True]
             }
kfold = KFold(n_splits=5, random_state=26,shuffle=True)
gs = GridSearchCV(surprise.KNNBasic, param_grid,measures=['rmse','mae'], cv=kfold)
# gs = GridSearchCV(surprise.KNNWithZScore, param_grid,measures=['rmse','mae'], cv=kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [55]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])
print(gs.best_score['mae'])
print(gs.best_params['mae'])

1.1982508756078385
{'k': 25, 'user_based': False}
0.8905608814281143
{'k': 25, 'user_based': False}


In [56]:
# Using SVD
param_grid = {  'n_epochs':[50],
                'n_factors':[100,50],
                'reg_all':np.linspace(0.001,1,5),
                'reg_all':np.linspace(0.01,0.8,5)
             }
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.SVD, param_grid, measures=['rmse','mae'], joblib_verbose=3, cv=kfold, n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  36 out of  50 | elapsed:    2.0s remaining:    0.7s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    2.1s finished


In [57]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])
print(gs.best_score['mae'])
print(gs.best_params['mae'])

1.1572631224641527
{'n_epochs': 50, 'n_factors': 50, 'reg_all': np.float64(0.01)}
0.8432303547067395
{'n_epochs': 50, 'n_factors': 50, 'reg_all': np.float64(0.01)}
